# Baseline: Popularity, ItemKNN, EASE

Thay đổi so với bản cũ:

- Lưới `k` của ItemKNN mở rộng đến 2000, `shrink` đến 100 và `lam` của EASE xuống 10 (lần chạy trước có giá trị tốt nhất nằm đúng biên lưới). Cảnh báo biên lưới nay có cho cả `k`, `shrink` và `lam`; nếu còn bật thì mở rộng lưới tiếp trước khi chạy hybrid.
- Thêm EASE làm baseline mạnh hơn, và biến thể "+pop" cộng một lượng rất nhỏ theo độ phổ biến để phá hòa giữa các item cùng điểm 0.
- Mọi số đo có khoảng tin cậy bootstrap 95%.
- Tham số tốt nhất theo valid được lưu vào `best_params.json` để notebook hybrid dùng đúng cấu hình này.

In [ ]:
import sys; sys.path.append("..")
import json
import numpy as np, pandas as pd
from src.config import CATEGORY, SPLITS, RESULTS
from src.models import Popularity, ItemKNN, EASE, PopFallback, build_matrix
from src.metrics import evaluate
from src.stats import bootstrap_metric

train, valid, test = [pd.read_csv(SPLITS / f"{n}.csv.gz") for n in ("train", "valid", "test")]
allx = pd.concat([train, valid, test])
user2idx = {u: i for i, u in enumerate(allx.user_id.unique())}
item2idx = {p: i for i, p in enumerate(allx.parent_asin.unique())}
X_tr = build_matrix(train, user2idx, item2idx)
X_trva = build_matrix(pd.concat([train, valid]), user2idx, item2idx)
print(CATEGORY, "| user:", len(user2idx), "| item:", len(item2idx),
      "| train/valid/test:", len(train), len(valid), len(test))

POP_EPS = 1e-4
GRID_K = (100, 200, 500, 1000, 2000)
GRID_SHRINK = (0, 10, 30, 100)
GRID_LAM = (10, 30, 50, 100, 200, 500)

## 1. Chọn tham số trên valid (fit train, lịch sử = train)

In [ ]:
rows = []
m, _ = evaluate(Popularity().fit(X_tr), X_tr, valid, user2idx, item2idx)
rows.append({"family": "Popularity", **m})

for k in GRID_K:
    for s in GRID_SHRINK:
        m, _ = evaluate(ItemKNN(k, s).fit(X_tr), X_tr, valid, user2idx, item2idx)
        rows.append({"family": "ItemKNN", "k": k, "shrink": s, **m})

ease_ok = True
for lam in GRID_LAM:
    try:
        m, _ = evaluate(EASE(lam).fit(X_tr), X_tr, valid, user2idx, item2idx)
    except ValueError as e:
        print("Bỏ qua EASE:", e); ease_ok = False; break
    rows.append({"family": "EASE", "lam": lam, **m})

val_df = pd.DataFrame(rows)
display(val_df[["family", "k", "shrink", "lam", "HR@10", "NDCG@10", "NDCG@20"]].round(4))
val_df.to_csv(RESULTS / "baselines_valid.csv", index=False)

best_knn = val_df[val_df.family == "ItemKNN"].sort_values("NDCG@10").iloc[-1]
best = {"knn": {"k": int(best_knn.k), "shrink": float(best_knn.shrink)}}
if best_knn.k in (min(GRID_K), max(GRID_K)):
    print("CẢNH BÁO: k tốt nhất nằm ở biên lưới, mở rộng GRID_K.")
if best_knn.shrink in (min(GRID_SHRINK), max(GRID_SHRINK)):
    print("CẢNH BÁO: shrink tốt nhất nằm ở biên lưới, mở rộng GRID_SHRINK.")
if ease_ok:
    best_ease = val_df[val_df.family == "EASE"].sort_values("NDCG@10").iloc[-1]
    best["ease"] = {"lam": float(best_ease.lam)}
    if best_ease.lam in (min(GRID_LAM), max(GRID_LAM)):
        print("CẢNH BÁO: lam tốt nhất nằm ở biên lưới, mở rộng GRID_LAM.")
print(best)
(RESULTS / "best_params.json").write_text(json.dumps(best, indent=2))

## 2. Chấm test một lần (fit train + valid)

Mỗi dòng có HR@10 và NDCG@10 kèm khoảng tin cậy 95% bootstrap.

In [ ]:
def fit_family(name):
    if name == "Popularity":
        return Popularity().fit(X_trva)
    if name.startswith("ItemKNN"):
        base = ItemKNN(best["knn"]["k"], best["knn"]["shrink"]).fit(X_trva)
    else:
        base = EASE(best["ease"]["lam"]).fit(X_trva)
    return PopFallback(base, X_trva, POP_EPS) if name.endswith("+pop") else base

families = ["Popularity", "ItemKNN", "ItemKNN+pop"] + (["EASE", "EASE+pop"] if ease_ok else [])
rows = []
for name in families:
    m, rk = evaluate(fit_family(name), X_trva, test, user2idx, item2idx)
    np.save(RESULTS / f"ranks_{name.replace('+', '_')}.npy", rk)
    h, hl, hh = bootstrap_metric(rk, 10, "hr")
    n, nl, nh = bootstrap_metric(rk, 10, "ndcg")
    rows.append({"model": name, "HR@10": h, "HR@10 lo": hl, "HR@10 hi": hh,
                 "NDCG@10": n, "NDCG@10 lo": nl, "NDCG@10 hi": nh, "n_eval": m["n_eval"]})
test_df = pd.DataFrame(rows)
display(test_df.round(4))
test_df.to_csv(RESULTS / "baselines_test.csv", index=False)

Nên đọc kết quả này trước khi sang hybrid: nếu ItemKNN và EASE chênh nhau rất xa, kết luận về sentiment cần được kiểm tra trên cả hai thuật toán nền. Notebook hybrid chạy cả hai.